# 0.3 Clean and Export the Recent Ru-AHO Dataset

This notebook prepares the recent Ru-catalyzed asymmetric olefin hydrogenation (AHO) literature dataset for external validation.

Input:
- `data/raw_data/ru_aho_2020_2025.csv`

Output:
- `data/recent_aho/recent_aho_dataset.csv`

The notebook intentionally does only one thing: clean the recent AHO table and export it. It does not generate metal-distribution summaries, figures, ChemProp files, or extra processed-data directories.

Publication-year information is attached after deduplication. The year column does not participate in deduplication; if multiple literature records match the same deduplicated reaction-condition row, the earliest available year is retained. DOI columns are not exported.


In [1]:
from pathlib import Path


def find_project_root(start=None):
    """Find the repository root from either the notebook folder or the project root."""
    if start is None:
        start = Path.cwd()
    start = Path(start).resolve()

    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "data").exists() or (candidate / "notebooks").exists():
            return candidate

    # Fallback for a notebook located at notebooks/0_data_preparation before data/ is created.
    if start.name == "0_data_preparation" and start.parent.name == "notebooks":
        return start.parents[1]
    return start


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_ROOT = DATA_ROOT / "raw_data"
RECENT_AHO_DATA_ROOT = DATA_ROOT / "recent_aho"

RECENT_AHO_RAW_PATH = RAW_DATA_ROOT / "ru_aho_2020_2025.csv"
RECENT_AHO_OUTPUT_PATH = RECENT_AHO_DATA_ROOT / "recent_aho_dataset.csv"

RECENT_AHO_DATA_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root: {}".format(PROJECT_ROOT))
print("Input : {}".format(RECENT_AHO_RAW_PATH))
print("Output: {}".format(RECENT_AHO_OUTPUT_PATH))


Project root: /home/syz/ru-mechanism-transfer
Input : /home/syz/ru-mechanism-transfer/data/raw_data/ru_aho_2020_2025.csv
Output: /home/syz/ru-mechanism-transfer/data/recent_aho/recent_aho_dataset.csv


## Clean and export the recent AHO table

The cleaned table follows the same modeling-oriented schema used by the transfer-domain files: core reaction SMILES, catalyst/condition fields, and `ddG`.


In [2]:
import re

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from rdkit import Chem
except ImportError:
    Chem = None
    print("Warning: RDKit is not available. SMILES strings will be stripped but not canonicalized.")

R_KCAL = 0.0019872  # kcal mol^-1 K^-1


# Known source-year hints for the three recent Ru-AHO literature records used here.
# These hints are used only to recover Reaction Year when an explicit year column is absent.
SOURCE_YEAR_HINTS = {
    "10.1021/acs.orglett.5c05118": 2026,
    "acs.orglett.5c05118": 2026,
    "5c05118": 2026,
    "10.1002/advs.202400621": 2024,
    "advs.202400621": 2024,
    "202400621": 2024,
    "10.1039/d2qo00448h": 2022,
    "d2qo00448h": 2022,
}


def clean_string(value):
    """Normalize missing string-like values to an empty string."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.lower() in ["nan", "none", "null"]:
        return ""
    return text


def first_existing_column(df, candidates):
    """Return the first existing column from a list of candidate names."""
    for col in candidates:
        if col in df.columns:
            return col
    return None


def require_any_column(df, candidates, field_name):
    """Return the first candidate column and raise a clear error if none is present."""
    col = first_existing_column(df, candidates)
    if col is None:
        raise KeyError(
            "Cannot build field '{}': none of these columns were found: {}".format(
                field_name, candidates
            )
        )
    return col


def standardize_smiles(smiles):
    """Return canonical isomeric SMILES when RDKit is available; otherwise return stripped text."""
    text = clean_string(smiles)
    if text == "":
        return ""
    if Chem is None:
        return text

    mol = Chem.MolFromSmiles(text)
    if mol is None:
        return ""
    try:
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    except Exception:
        return ""


def cleaned_series_from_aliases(df, candidates, default=""):
    """Return a cleaned string series from the first available candidate column."""
    col = first_existing_column(df, candidates)
    if col is None:
        return pd.Series([default] * len(df), index=df.index)
    return df[col].map(clean_string)


def numeric_series_from_aliases(df, candidates):
    """Return a numeric series from the first available candidate column, else NaN."""
    col = first_existing_column(df, candidates)
    if col is None:
        return pd.Series([np.nan] * len(df), index=df.index)
    return pd.to_numeric(df[col], errors="coerce")


def infer_year_from_source_text(row):
    """Infer the publication year from DOI/source/title text when no explicit year is available."""
    pieces = []
    for value in row:
        text = clean_string(value)
        if text:
            pieces.append(text.lower())
    merged = " | ".join(pieces)
    if not merged:
        return np.nan

    for key, year in SOURCE_YEAR_HINTS.items():
        if key.lower() in merged:
            return year

    # Conservative fallback: accept only years in the recent-AHO external-validation range.
    matches = re.findall(r"(?<!\d)(202[0-6])(?!\d)", merged)
    if matches:
        return int(min(matches))
    return np.nan


def publication_year_series_from_raw(df):
    """Recover publication year from explicit year columns, then from source/DOI-like text."""
    explicit_year = numeric_series_from_aliases(
        df,
        [
            "Reaction Year",
            "Year",
            "Publication Year",
            "Published Year",
            "Reference Year",
            "Literature Year",
            "Article Year",
            "Pub Year",
        ],
    )

    source_like_cols = [
        col for col in df.columns
        if any(token in col.lower() for token in ["doi", "year", "source", "reference", "literature", "article", "title", "journal"])
    ]
    if source_like_cols:
        inferred_year = df[source_like_cols].apply(infer_year_from_source_text, axis=1)
    else:
        inferred_year = pd.Series([np.nan] * len(df), index=df.index)

    year = explicit_year.copy()
    year = year.where(year.notna(), inferred_year)
    year = pd.to_numeric(year, errors="coerce")

    # Keep clean publication-year values for this recent external-validation set.
    year = year.where((year >= 2020) & (year <= 2026), np.nan)
    return year


def attach_publication_year_after_dedup(dedup_df, source_df, match_cols):
    """
    Attach Reaction Year after deduplication.

    The year column is deliberately kept outside the deduplication key. If several raw
    literature records map to the same deduplicated reaction-condition row, the earliest
    available publication year is retained.
    """
    year_source = source_df[match_cols + ["_source_year", "_raw_row_order"]].copy()
    year_source["_year_sort"] = year_source["_source_year"].fillna(np.inf)

    year_lookup = (
        year_source
        .sort_values(
            by=match_cols + ["_year_sort", "_raw_row_order"],
            ascending=True,
            na_position="last",
        )
        .drop_duplicates(subset=match_cols, keep="first")
        [match_cols + ["_source_year"]]
        .rename(columns={"_source_year": "Reaction Year"})
    )

    result = dedup_df.drop(columns=["Reaction Year"], errors="ignore").merge(
        year_lookup,
        on=match_cols,
        how="left",
        validate="one_to_one",
        sort=False,
    )

    result["Reaction Year"] = (
        pd.to_numeric(result["Reaction Year"], errors="coerce")
        .round()
        .astype("Int64")
    )

    n_with_year = int(result["Reaction Year"].notna().sum())
    print("Rows with recovered publication year: {}/{}".format(n_with_year, len(result)))
    return result


def merge_smiles_columns(df, columns):
    """Concatenate non-empty SMILES fields with '.' for multi-component entries."""
    existing = [col for col in columns if col in df.columns]
    if not existing:
        return pd.Series([""] * len(df), index=df.index)

    def _merge(row):
        parts = []
        for col in existing:
            text = clean_string(row[col])
            if text:
                parts.append(text)
        return ".".join(parts)

    return df.apply(_merge, axis=1)


def merged_or_single_smiles(df, canonical_candidates, multi_component_candidates):
    """Use an existing canonical column if present; otherwise merge multi-component columns."""
    col = first_existing_column(df, canonical_candidates)
    if col is not None:
        return df[col].map(clean_string)
    return merge_smiles_columns(df, multi_component_candidates)


def compute_ddg_from_ee(ee_series, temperature_c_series):
    """Convert ee (%) to ddG (kcal/mol) using dG = RT ln(er)."""
    ee = pd.to_numeric(ee_series, errors="coerce")
    temp_c = pd.to_numeric(temperature_c_series, errors="coerce")
    temp_k = temp_c.fillna(25.0) + 273.15

    safe_ee = ee.clip(lower=-99.999, upper=99.999).abs()
    er = (1 + safe_ee / 100.0) / (1 - safe_ee / 100.0)
    return R_KCAL * temp_k * np.log(er)


def save_csv(df, path):
    """Save a dataframe after creating the parent directory."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print("Saved: {}  shape={}".format(path, df.shape))


if not RECENT_AHO_RAW_PATH.exists():
    raise FileNotFoundError("Cannot find recent AHO raw table: {}".format(RECENT_AHO_RAW_PATH))

recent_aho_raw = pd.read_csv(RECENT_AHO_RAW_PATH)
print("Raw recent AHO shape: {}".format(recent_aho_raw.shape))

publication_year = publication_year_series_from_raw(recent_aho_raw)

reactant_col = require_any_column(
    recent_aho_raw,
    [
        "Reactant SMILES",
        "Reactant SMILES 1",
        "Substrate SMILES",
        "Substrate SMILES 1",
        "Olefin SMILES",
    ],
    "Reactant SMILES",
)
product_col = require_any_column(
    recent_aho_raw,
    [
        "Product SMILES",
        "Product SMILES 1",
        "Hydrogenated Product SMILES",
        "Product",
    ],
    "Product SMILES",
)

recent_aho = pd.DataFrame(index=recent_aho_raw.index)
recent_aho["Reactant SMILES"] = recent_aho_raw[reactant_col].apply(standardize_smiles)
recent_aho["Product SMILES"] = recent_aho_raw[product_col].apply(standardize_smiles)
recent_aho["Reaction Year"] = publication_year
recent_aho["Metal"] = cleaned_series_from_aliases(
    recent_aho_raw,
    ["Metal", "Catalyst Central Metal 1", "Catalyst Metal", "Metal Center", "metal"],
)
recent_aho["Ligand SMILES"] = merged_or_single_smiles(
    recent_aho_raw,
    ["Ligand SMILES", "Catalyst Ligand SMILES", "ligand_smiles"],
    [
        "Catalyst Ligand SMILES 1",
        "Catalyst Ligand SMILES 2",
        "Ligand SMILES 1",
        "Ligand SMILES 2",
    ],
)
recent_aho["Solvent SMILES"] = merged_or_single_smiles(
    recent_aho_raw,
    ["Solvent SMILES", "solvent_smiles"],
    [
        "Solvent SMILES 1",
        "Solvent SMILES 2",
        "Solvent SMILES 3",
        "Solvent 1 SMILES",
        "Solvent 2 SMILES",
    ],
)
recent_aho["Additive SMILES"] = merged_or_single_smiles(
    recent_aho_raw,
    ["Additive SMILES", "additive_smiles"],
    [
        "Additive SMILES 1",
        "Additive SMILES 2",
        "Additive SMILES 3",
        "Base SMILES",
        "Acid SMILES",
    ],
)
recent_aho["Pressure/atm"] = numeric_series_from_aliases(
    recent_aho_raw,
    ["Pressure/atm", "Step Pressure (atm) 1", "Pressure (atm)", "Pressure", "H2 Pressure/atm"],
)
recent_aho["Temperature/C"] = numeric_series_from_aliases(
    recent_aho_raw,
    ["Temperature/C", "Step Temperature (C) 1", "Temperature (C)", "Temperature", "T/C"],
)
recent_aho["S/C"] = numeric_series_from_aliases(
    recent_aho_raw,
    ["S/C", "Substrate/Catalyst", "Substrate to Catalyst", "S_to_C", "SC"],
)

# If S/C is not explicitly present, derive it from catalyst amount when available.
if recent_aho["S/C"].isna().all():
    catalyst_amount = numeric_series_from_aliases(
        recent_aho_raw,
        [
            "Catalyst Ammount (eq) 1",
            "Catalyst Amount (eq) 1",
            "Catalyst Amount (eq)",
            "Catalyst Loading (eq)",
        ],
    )
    if not catalyst_amount.isna().all():
        recent_aho["S/C"] = 1.0 / catalyst_amount.replace(0, np.nan)

# Prefer an existing ddG column. If absent, derive ddG from ee and temperature.
ddg_col = first_existing_column(recent_aho_raw, ["ddG", "DeltaDeltaG", "Delta Delta G", "dG"])
if ddg_col is not None:
    recent_aho["ddG"] = pd.to_numeric(recent_aho_raw[ddg_col], errors="coerce")
else:
    ee_col = require_any_column(
        recent_aho_raw,
        ["ee", "ee/%", "ee (%)", "Enantiomeric Excess", "Enantiomeric Excess (%)", "Product ee (%) 1"],
        "ddG or ee",
    )
    recent_aho["ddG"] = compute_ddg_from_ee(recent_aho_raw[ee_col], recent_aho["Temperature/C"])


# Normalize string columns after construction.
for col in ["Reactant SMILES", "Product SMILES", "Metal", "Ligand SMILES", "Solvent SMILES", "Additive SMILES"]:
    recent_aho[col] = recent_aho[col].map(clean_string)

# Remove rows without core reaction information.
before_required = len(recent_aho)
recent_aho = recent_aho[
    (recent_aho["Reactant SMILES"] != "")
    & (recent_aho["Product SMILES"] != "")
    & recent_aho["ddG"].notna()
].copy()
print("Rows before required-field filtering: {}".format(before_required))
print("Rows after  required-field filtering: {}".format(len(recent_aho)))

# Keep publication-year information outside the deduplication key.
# This preserves the reaction-condition deduplication behavior while still allowing the
# final table to indicate which recent literature year each retained row comes from.
dedup_key_cols = [col for col in recent_aho.columns if col not in ["ddG", "Reaction Year"]]
recent_aho_year_source = recent_aho.copy()
recent_aho_year_source["_source_year"] = pd.to_numeric(recent_aho_year_source["Reaction Year"], errors="coerce")
recent_aho_year_source["_raw_row_order"] = recent_aho_year_source.index.to_numpy()

# Deduplicate identical reaction-condition rows, retaining the largest ddG value.
recent_aho = (
    recent_aho
    .sort_values(by="ddG", ascending=False, na_position="last")
    .drop_duplicates(subset=dedup_key_cols, keep="first")
    .reset_index(drop=True)
)
print("Rows after deduplication: {}".format(len(recent_aho)))

# Attach Reaction Year only after deduplication. If several source rows map to the same
# deduplicated reaction-condition row, keep the earliest publication year.
recent_aho = attach_publication_year_after_dedup(
    dedup_df=recent_aho,
    source_df=recent_aho_year_source,
    match_cols=dedup_key_cols,
)

# Keep a stable, modeling-oriented column order and do not export DOI information.
final_cols = [
    "Reactant SMILES",
    "Product SMILES",
    "Reaction Year",
    "Metal",
    "Ligand SMILES",
    "Solvent SMILES",
    "Additive SMILES",
    "Pressure/atm",
    "Temperature/C",
    "S/C",
    "ddG",
]
recent_aho = recent_aho[final_cols].copy()

display(recent_aho.head())
save_csv(recent_aho, RECENT_AHO_OUTPUT_PATH)


Raw recent AHO shape: (154, 90)
Rows before required-field filtering: 154
Rows after  required-field filtering: 154
Rows after deduplication: 103
Rows with recovered publication year: 103/103


,Reactant SMILES,Product SMILES,Reaction Year,Metal,Ligand SMILES,Solvent SMILES,Additive SMILES,Pressure/atm,Temperature/C,S/C,ddG
0,C=C(CC(=O)c1ccccc1)C(=O)O,C[C@@H](CC(=O)c1ccccc1)C(=O)O,2024,Ru,CC([C@H]1COC(=N1)C12C3[Ru]4567891(C2(C7C34)P(c...,CCO,[Na+].[O-1]C(=O)O,20.0,25.0,100.0,3.136197
1,C=C(CC(=O)c1ccc(C)cc1)C(=O)O,Cc1ccc(C(=O)C[C@H](C)C(=O)O)cc1,2026,Ru,CC([C@H]1COC(=N1)C12C3[Ru]4567891(C2(C7C34)P(c...,CCO,[Na+].[O-1]C(=O)O,20.0,25.0,100.0,3.136197
2,C=C(CC(=O)c1ccc(CC)cc1)C(=O)O,CCc1ccc(C(=O)C[C@H](C)C(=O)O)cc1,2026,Ru,CC([C@H]1COC(=N1)C12C3[Ru]4567891(C2(C7C34)P(c...,CCO,[Na+].[O-1]C(=O)O,20.0,25.0,100.0,3.136197
3,C=C(CC(=O)c1ccc(C(C)C)cc1)C(=O)O,CC(C)c1ccc(C(=O)C[C@H](C)C(=O)O)cc1,2026,Ru,CC([C@H]1COC(=N1)C12C3[Ru]4567891(C2(C7C34)P(c...,CCO,[Na+].[O-1]C(=O)O,20.0,25.0,100.0,3.136197
4,C=C(CC(=O)c1ccc(C(C)(C)C)cc1)C(=O)O,C[C@@H](CC(=O)c1ccc(C(C)(C)C)cc1)C(=O)O,2026,Ru,CC([C@H]1COC(=N1)C12C3[Ru]4567891(C2(C7C34)P(c...,CCO,[Na+].[O-1]C(=O)O,20.0,25.0,100.0,3.136197


Saved: /home/syz/ru-mechanism-transfer/data/recent_aho/recent_aho_dataset.csv  shape=(103, 11)
